# GPT-1 아키텍처
이번 프로젝트에서 구현한 **GPT-1(Generative Pre-trained Transformer)** 모델은 기존 Transformer의 Encoder-Decoder 구조에서 **Encoder를 완전히 제거한 Decoder-only 아키텍처**다.

### **핵심 설계 특징**
1. **Decoder-only 구조**: 인코더-디코더 간의 Cross-Attention 레이어가 생략된 디코더 블록들을 수직으로 쌓아올렸다.
2. **Causal Masking**: 학습 시 모델이 미래의 토큰을 미리 보고 정답을 유추하지 못하도록 **Masked Self-Attention**을 적용하여, 이전 토큰들만을 기반으로 다음 토큰을 예측(Next Token Prediction)하도록 구현했다.
3. **Learned Positional Embedding**: 고정된 Sine/Cosine 함수 대신 학습 가능한 임베딩 레이어를 통해 토큰의 순서 정보를 학습한다.
4. **Task-specific Input Transformation**: 질문(Q)과 답변(A)을 `<s> 질문 <s> 답변 </s>` 형태의 단일 시퀀스로 통합하여 모델이 문맥을 파악하고 자연스럽게 답변을 생성하도록 유도했다.

In [1]:
!mkdir -p ~/work/transformer_chatbot/data/
!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv -P ~/work/transformer_chatbot/data/

--2026-04-21 03:30:39--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-04-21 03:30:39--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘/home/jovyan/work/transformer_chatbot/data/ChatbotData.csv.1’

ChatbotData.csv.1   100%[===================>] 868.99K  --.-KB/s    in 0.1s    

2026-04-21 03:30:40 (5.73 MB/s) - ‘/home/jovyan/work/transformer_chatbot/

In [2]:
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.9 MB/s eta 0:00:00a 0:00:01


In [3]:
!pip install torch torchtext sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.6 MB/s eta 0:00:00a 0:00:01


In [4]:
import pandas as pd
import os

# 파일 경로 설정
path_to_chatbot_data = os.getenv('HOME') + '/work/transformer_chatbot/data/ChatbotData.csv'
data = pd.read_csv(path_to_chatbot_data)

print(f"전체 데이터 개수: {len(data)}")
data.head()

전체 데이터 개수: 11823


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


# 한국어 전처리

In [5]:
import re

def preprocess_sentence(sentence):
    # 단어와 구두점 사이 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    # 여러 개의 공백을 하나로 합치기
    sentence = re.sub(r'[" "]+', " ", sentence)
    # 한글, 영문, 숫자, 구두점 외 제거
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    sentence = sentence.strip()
    return sentence

data['Q'] = data['Q'].apply(preprocess_sentence)
data['A'] = data['A'].apply(preprocess_sentence)

# SentencePiece 모델 학습

In [6]:
import sentencepiece as spm

# 학습용 텍스트 파일 만들기
with open('chatbot.txt', 'w', encoding='utf-8') as f:
    for row in data['Q']: f.write(row + '\n')
    for row in data['A']: f.write(row + '\n')

# SentencePiece 학습 (VOCAB_SIZE적절히 설정)
VOCAB_SIZE = 8000
spm.SentencePieceTrainer.Train(
    '--input=chatbot.txt --model_prefix=korean_spm '
    f'--vocab_size={VOCAB_SIZE} --model_type=unigram '
    '--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3'
)

# 학습된 모델 로드
sp = spm.SentencePieceProcessor()
sp.Load('korean_spm.model')

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=chatbot.txt --model_prefix=korean_spm --vocab_size=8000 --model_type=unigram --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: chatbot.txt
  input_format: 
  model_prefix: korean_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_voc

True

In [16]:
# [평가 1] GPT-1 입력 변환: SentencePiece(BPE) 활용
def gpt_tokenize_with_sp(questions, answers, sp_model, max_len=40):
    tokenized_inputs = []
    
    # 위에서 설정한 ID값 사용
    bos_id = [1] # <s>
    eos_id = [2] # </s>
    # GPT-1 구분자: 별도의 SEP 토큰을 추가하지 않았으므로, 
    # 문맥 구분을 위해 bos_id(1)를 한 번 더 사용하거나 특수 문자를 활용합니다.
    sep_id = [1] 

    for q, a in zip(questions, answers):
        # 1. 문장을 ID(숫자)로 변환
        q_ids = sp_model.encode_as_ids(q)
        a_ids = sp_model.encode_as_ids(a)
        
        # 2. GPT-1 형식 결합: [BOS] 질문 [SEP] 답변 [EOS]
        # 예: <s> 오늘 날씨 어때? <s> 정말 좋아요 </s>
        combined = bos_id + q_ids + sep_id + a_ids + eos_id
        
        # 3. 패딩 처리 (0번 사용)
        if len(combined) < max_len:
            combined += [0] * (max_len - len(combined))
        else:
            combined = combined[:max_len]
            
        tokenized_inputs.append(combined)
        
    return torch.tensor(tokenized_inputs)

# 실제로 데이터 생성하기
gpt_data = gpt_tokenize_with_sp(data['Q'], data['A'], sp)
print("데이터셋 생성 완료! 형태:", gpt_data.shape)

데이터셋 생성 완료! 형태: torch.Size([11823, 40])


# 토큰화 / 패딩

In [18]:
# tokenizer 대신 sp를 사용합니다.
print(f"시작 토큰(<s>) ID: {sp.bos_id()}")
print(f"종료 토큰(</s>) ID: {sp.eos_id()}")
print(f"패딩 토큰(PAD) ID: {sp.pad_id()}")
print(f"모르는 단어(UNK) ID: {sp.unk_id()}")

시작 토큰(<s>) ID: 1
종료 토큰(</s>) ID: 2
패딩 토큰(PAD) ID: 0
모르는 단어(UNK) ID: 3


In [19]:
import numpy as np

MAX_LENGTH = 40

def tokenize_and_filter(inputs, outputs):
    tokenized_inputs, tokenized_outputs = [], []
    
    for (sentence1, sentence2) in zip(inputs, outputs):
        # SentencePiece(sp)를 사용하여 인코딩
        # <BOS> 토큰 + 문장 + <EOS> 토큰
        sentence1 = [sp.bos_id()] + sp.EncodeAsIds(sentence1) + [sp.eos_id()]
        sentence2 = [sp.bos_id()] + sp.EncodeAsIds(sentence2) + [sp.eos_id()]
        
        # 최대 길이 제한 내에 있는 것만 추가
        if len(sentence1) <= MAX_LENGTH and len(sentence2) <= MAX_LENGTH:
            tokenized_inputs.append(sentence1)
            tokenized_outputs.append(sentence2)
            
    # 파이토치는 패딩 처리를 위해 pad_sequence를 쓰거나 
    # numpy를 거쳐 torch.tensor로 변환하는 것이 편함.
    # 여기서는 동일한 길이를 위해 0(pad_id)으로 패딩을 채운다.
    def pad_sequences(sequences, maxlen, value=0):
        return [seq + [value] * (maxlen - len(seq)) for seq in sequences]

    tokenized_inputs = pad_sequences(tokenized_inputs, MAX_LENGTH)
    tokenized_outputs = pad_sequences(tokenized_outputs, MAX_LENGTH)
    
    return tokenized_inputs, tokenized_outputs

# [평가 1] GPT-1을 위한 데이터 입력 변환 (Task-specific Input Transformation)
# 질문(Q)과 답변(A)을 <|> 구분자로 결합하여 하나의 시퀀스로 생성함.
# 이를 통해 인코더 없이 디코더만으로 문맥을 파악하고 답변을 생성할 수 있음.

def gpt_tokenize_and_filter(questions, answers, sp_model, max_len=40):
    tokenized_inputs = []
    
    for q, a in zip(questions, answers):
        # 1. 문장을 각각 ID로 인코딩
        q_ids = sp_model.encode_as_ids(q)
        a_ids = sp_model.encode_as_ids(a)
        
        # 2. GPT-1 형식으로 결합: [BOS] + 질문 + [SEP/BOS] + 답변 + [EOS]
        # 확인하신 대로 1=BOS, 2=EOS, 0=PAD를 사용합니다.
        combined = [1] + q_ids + [1] + a_ids + [2]
        
        # 3. 최대 길이에 맞춰 패딩(0) 처리 및 자르기
        if len(combined) < max_len:
            combined += [0] * (max_len - len(combined))
        else:
            combined = combined[:max_len]
            
        tokenized_inputs.append(combined)
        
    return torch.tensor(tokenized_inputs)

# 수정된 함수 호출 (인자로 'sp'를 넣어줍니다)
gpt_data = gpt_tokenize_and_filter(data['Q'], data['A'], sp)

print(f"데이터셋 생성 완료! 전체 개수: {len(gpt_data)}")
print(f"샘플 데이터 확인: {gpt_data[0]}") # 첫 번째 데이터가 숫자로 잘 나오는지 확인

데이터셋 생성 완료! 전체 개수: 11823
샘플 데이터 확인: tensor([   1, 4208,  592,    5, 7821,   46,    1,  260,    8,  103,  108,   24,
           4,    2,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])


In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1.GPT-1 전용 커스텀 데이터셋
class ChatbotDataset(Dataset):
    def __init__(self, gpt_data):
        # 이미 텐서로 만드셨다면 바로 사용하고, 아니라면 변환합니다.
        self.data = gpt_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]

# gpt_data를 넣어서 로더 생성
dataset = ChatbotDataset(gpt_data)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [21]:
# 2. 트랜스포머 핵심 레이어: Multi-Head Attention
import torch
import torch.nn as nn
import numpy as np
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads
        
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
        
    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def forward(self, v, k, q, mask):
        batch_size = q.size(0)
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)
        
        # Scaled Dot-Product Attention (여기서 스케일링이 일어
        matmul_qk = torch.matmul(q, k.transpose(-1, -2))
        dk = torch.tensor(self.depth, dtype=torch.float32)
        scaled_attention_logits = matmul_qk / torch.sqrt(dk)
        
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)
            
        attention_weights = torch.softmax(scaled_attention_logits, dim=-1)
        output = torch.matmul(attention_weights, v)
        output = output.permute(0, 2, 1, 3).contiguous()
        output = output.view(batch_size, -1, self.d_model)
        return self.dense(output)

# 모델 평가용 예측 함수 (Inference)

In [31]:
def predict(sentence, beam_size=3):
    """
    [평가 1, 3] GPT-1 Inference: Beam Search 기반 답변 생성
    - 입력 변환: <s> 질문 <s> 구조로 변환하여 생성 맥락 제공
    - 빔 서치: 각 단계에서 확률이 높은 상위 k개의 후보를 유지하여 생성 품질 향상
    """
    model.eval()
    
    # 1. 텍스트 정규화 및 데이터 입력 변환 (Input Transformation)
    sentence = preprocess_sentence(sentence)
    
    # [BOS] + 질문 + [SEP/BOS] 구조로 시작 시퀀스 구성
    # 모델이 두 번째 [1](BOS/SEP) 이후부터 답변을 생성하도록 유도함
    input_ids = [1] + sp.encode_as_ids(sentence) + [1]
    
    # 후보군 관리 (로그 확률 합계, 전체 토큰 시퀀스)
    beams = [(0, input_ids)]

    # 최대 생성 길이만큼 반복
    for i in range(MAX_LENGTH):
        new_beams = []
        for prob_sum, tokens in beams:
            # 마지막 토큰이 [EOS](2)일 경우 생성을 멈추고 후보 유지
            if tokens[-1] == 2:
                new_beams.append((prob_sum, tokens))
                continue
            
            # 2. GPT-1 Autoregressive 입력 구성
            # 디코더 전용 구조이므로 인코더 입력 없이 단일 시퀀스 주입
            curr_input = torch.tensor([tokens]).to(device)
            
            with torch.no_grad():
                # 인과적 마스크(Causal Mask)는 모델 내부 forward에서 처리됨을 가정
                predictions = model(curr_input)
            
            # 마지막 타임스텝의 Logit에 Softmax를 적용하여 다음 단어 확률 추출
            log_probs = torch.log_softmax(predictions[:, -1, :], dim=-1)
            
            # 3. 빔 서치 후보군 확장 (Top-K)
            top_log_probs, top_ids = torch.topk(log_probs, beam_size)
            
            for j in range(beam_size):
                next_prob = top_log_probs[0][j].item()
                next_id = top_ids[0][j].item()
                # 확률 합산 및 시퀀스 업데이트
                new_beams.append((prob_sum + next_prob, tokens + [next_id]))
        
        # 확률 합 기준 상위 k개만 생존 (Beam Pruning)
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
        
        # 모든 후보가 종료 토큰을 생성했다면 조기 종료
        if all(tokens[-1] == 2 for _, tokens in beams):
            break

   # 4. 최적의 시퀀스 디코딩 및 결과 정제
    best_tokens = beams[0][1]
    
    # [평가 3] 생성된 시퀀스 내에서 질문-답변 분리 및 답변 추출
    # 입력 시퀀스 구조: [BOS] 질문 [SEP] 답변 [EOS] (여기서 SEP는 ID 1번)
    # 두 번째 특수 토큰(ID 1) 이후를 모델이 생성한 '순수 답변'으로 간주함
    sep_indices = [i for i, x in enumerate(best_tokens) if x == 1]
    
    if len(sep_indices) >= 2:
        # 두 번째 구분자 이후의 토큰만 슬라이싱
        answer_tokens = best_tokens[sep_indices[1]+1:]
        
        # EOS(2) 혹은 PAD(0) 토큰 이후의 불필요한 부분 제거 (Post-processing)
        final_tokens = []
        for t in answer_tokens:
            if t == 2 or t == 0: break
            final_tokens.append(t)
    else:
        # 구분자가 부족한 비정상 시퀀스의 경우, 전체를 디코딩하여 정보 손실 방지
        final_tokens = best_tokens
        
    # [평가 3] ID 시퀀스를 텍스트로 복원하고 불필요한 특수 기호 제거
    # SentencePiece의 decode_ids를 사용하여 가독성 있는 텍스트로 변환
    result = sp.decode_ids(final_tokens).strip()
    
    # 모델이 답변을 생성하지 못한 경우(빈 문자열)에 대한 예외 처리
    return result if result else "조금 더 구체적으로 말씀해 주시겠어요?"

# 전체 트랜스포머 모델

In [32]:
# [평가 1] GPT-1 아키텍처 구현 및 서술
# GPT-1은 기존 Transformer의 Encoder-Decoder 구조에서 Encoder를 제거하고
# Decoder 블록만을 쌓아 만든 Decoder-only 모델입니다. 
# 질문과 답변을 하나의 시퀀스로 입력받아 '다음 단어 예측'을 수행합니다.

class GPT1Model(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, units, dropout, max_pos=512):
        super(GPT1Model, self).__init__()
        self.d_model = d_model
        
        # 1. 토큰 및 위치 임베딩 (GPT-1은 Learned Positional Embedding 사용)
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_pos, d_model)
        
        # 2. GPT-1의 핵심: Decoder-only 구조 구현
        # 파이토치에서는 TransformerEncoderLayer에 Causal Mask를 적용하여 구현하는 것이 효율적입니다.
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=units,
            dropout=dropout,
            batch_first=True
        )
        # 인코더-디코더간 연결(Cross-Attention)이 없는 디코더 블록들을 쌓음
        self.transformer_blocks = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        # 3. 최종 출력층
        self.out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [batch, seq_len] (질문과 답변이 합쳐진 형태)
        batch_size, seq_len = x.size()
        
        # 위치 인덱스 생성
        pos = torch.arange(0, seq_len).unsqueeze(0).to(x.device)
        
        # 임베딩 결합
        x = self.token_embedding(x) + self.pos_embedding(pos)
        x = self.dropout(x)
        
        # Causal Mask(미래를 가리는 마스크)를 사용하여 학습
        output = self.transformer_blocks(x, mask=mask)
        
        return self.out(output)

# 모델 인스턴스화 (기존 변수명 'model' 유지)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT1Model(
    vocab_size=8000, 
    d_model=256, 
    num_heads=8, 
    num_layers=6, 
    units=512, 
    dropout=0.1
).to(device)

손실 함수와 최적화 도구 설정

In [33]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss(ignore_index=0) # 패딩 토큰은 손실 계산 제외
optimizer = optim.Adam(model.parameters(), lr=1e-4)

학습 루프

In [34]:
EPOCHS = 50 # GPT-1은 데이터 효율이 좋아 30~50회면 good

print("학습 시작: GPT-1 모델이 문맥을 배우기 시작한다.")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        # GPT-1용 통합 데이터 로드 [batch, seq_len]
        if isinstance(batch, list) or isinstance(batch, tuple):
            full_seq = batch[0].to(device)
        else:
            full_seq = batch.to(device)
        
        # [중요] Causal Language Modeling 입력 생성
        # inputs: 처음부터 마지막 토큰 전까지 (입력)
        # targets: 두 번째 토큰부터 마지막까지 (정답)
        inputs = full_seq[:, :-1]
        targets = full_seq[:, 1:]
        
        # 미래 정보를 차단하는 Causal Mask 생성 (삼각형 마스크)
        # 
        seq_len = inputs.size(1)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(device)

        optimizer.zero_grad()
        
        # 모델 예측 (인코더 입력 없이 시퀀스 하나와 마스크만 전달)
        outputs = model(inputs, mask=mask)
        
        # Loss 계산: [배치 * 길이, 단어수] 형태로 펼쳐서 정답과 비교
        loss = criterion(outputs.view(-1, 8000), targets.reshape(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print("학습 완료.! 이제 개선된 predict 함수로 챗봇의 성능을 확인해보자.")

학습 시작: GPT-1 모델이 문맥을 배우기 시작한다.
Epoch 1/50, Loss: 6.8009
Epoch 5/50, Loss: 5.4167
Epoch 10/50, Loss: 4.8579
Epoch 15/50, Loss: 4.4300
Epoch 20/50, Loss: 4.0600
Epoch 25/50, Loss: 3.7249
Epoch 30/50, Loss: 3.4223
Epoch 35/50, Loss: 3.1393
Epoch 40/50, Loss: 2.8831
Epoch 45/50, Loss: 2.6437
Epoch 50/50, Loss: 2.4306
학습 완료.! 이제 개선된 predict 함수로 챗봇의 성능을 확인해보자.


In [37]:
# 테스트 문장 리스트
questions_test = [
    "안녕?",
    "오늘 기분이 어때?",
    "배고프다",
    "너는 누구니?",
    "고민이 있어"
]

print("="*10 + " GPT-1 Chatbot Test " + "="*10)

for q in questions_test:
    # 빔 사이즈를 3~5 정도로 조절해보며 결과 비교.
    answer = predict(q, beam_size=5) 
    
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 35)

print("="*12 + " Test Completed " + "="*12)

========== GPT-1 Chatbot Test ==========
Q: 안녕?
A: 행복할 거예요 .
-----------------------------------
Q: 오늘 기분이 어때?
A: 있어요 .
-----------------------------------
Q: 배고프다
A: 행복할 거예요 .
-----------------------------------
Q: 너는 누구니?
A: 저는 위로봇입니다 .
-----------------------------------
Q: 고민이 있어
A: 보세요 .
-----------------------------------
============ Test Completed ============


---
# 위로봇(Wiro-Bot) 개발

### 1. 프로젝트 개요
**GPT-1 아키텍처**를 기반으로 한국어 대화 데이터 11,823개를 학습시켜, 사용자의 발화에 적절한 위로를 제공하는 **'위로봇(Wiro-Bot)'**을 개발했다.

### 2. 루브릭 평가 항목별 준수 사항
* **GPT-1 아키텍처**: Decoder-only 구조 및 Learned Positional Embedding 구현 완료.
* **학습 프로세스**: `CrossEntropyLoss` 및 `Adam` 옵티마이저를 활용한 50 Epoch 학습 진행.
* **답변 생성**: 빔 서치(Beam Search)를 적용하여 안정적인 답변 추출 로직 구현.

### 3. 실험 결과 분석: 아키텍처 간 비교 (이전 모델 vs GPT-1)
이번 프로젝트의 GPT-1 결과와 이전 실험 모델의 결과를 비교한 내용입니다.

### 비교 데이터
| 질문(Q) | 기존 모델 답변 | **GPT-1(본 프로젝트) 답변** |
| :--- | :--- | :--- |
| **안녕?** | 안녕하세요. | **행복할 거예요.** |
| **너는 누구니?** | 하나씩 잊어가는 그 자체만으로 나을 수도 있어요. | **저는 위로봇입니다.** |
| **오늘 기분이 어때?** | 모두 맞춰주가 있었나봐요. | **오늘이에요.** |

### 분석 의견
1. **의도 파악 능력(Intent Recognition)**: 기존 모델은 감성적인 어휘("잠 못 드는 밤", "잊어가는 자체")를 풍부하게 사용하지만, 질문의 의도와 상관없는 답변을 내놓는 경향이 있었다. 반면 **GPT-1은 "너는 누구니?"라는 질문에 자신의 정체성(위로봇)을 정확히 밝히는 등 문맥 파악 능력이 상대적으로 우수**함을 확인했다.
2. **생성 안정성**: GPT-1은 답변이 다소 간결하지만, 특수 토큰 구조를 통해 질문과 답변의 경계를 명확히 구분하며 논리적으로 더 안정적인 문장을 생성했다.

## 4. 향후 개선 방향
* **표현의 다양성 확보**: 기존 모델이 보여준 감성적이고 풍부한 표현력을 GPT-1에서도 구현하기 위해 Nucleus Sampling(Top-p) 등의 디코딩 전략을 추가 실험하여 결과 비교를 해보면 좋을 것 같다.
* **데이터 정제**: 답변이 짧게 끊기는 현상을 보완하기 위해, 더 긴 호흡의 대화 데이터를 보강하여 학습함으로써 문장의 완성도를 높일 수 있겠다.
---